In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from modeling.dataloader import INSPIRE
from modeling.model import *
from modeling.utils import Logger

from tqdm import tqdm
import numpy as np
import json
import random
import pandas as pd
import os
import datetime

from sklearn.metrics import accuracy_score, recall_score, \
                            precision_score, f1_score, roc_auc_score, \
                            average_precision_score


In [ ]:
# 初始化 Logger
task_name = 'hr'
logger = Logger(save_dir='checkpoints', model_name=task_name + "_intraop", resume=True)


In [ ]:
y_index = 1
device = torch.device("cuda:1")
input_size_list = [37 + 36, 29 + 16, 71 * 2, 53-4]
hidden_size_list = [256, 256, 256, 256]
output_size_list = [3]
type_list = ["cat"]
model_list = ["lstm", "lstm", "lstm", "mlp"]


In [ ]:
ds = pd.read_csv("data/operation_.csv", header=0)
train_datasets = [INSPIRE(
        "data/all_op_id/",
        ds[(ds['dataset'] == 1)],
        id_col="op_id",
        param_path="./param_folder"
    )]


valid_datasets = [INSPIRE(
        "data/all_op_id/",
        ds[(ds['dataset'] == 3)],
        id_col="op_id",
        param_path="./param_folder"
    )]


In [ ]:
model = PredModel(input_size_list, hidden_size_list, model_list, output_size_list, type_list)
model = logger.load_best_model(model)
model = model.to(device)


In [ ]:
# -------- 训练参数 ------------
EPOCHS = 100
lr = 0.001
weight_decay = 0.001
batch_size = 300
batch_size_val = 300
best_auc = 0.5
no_improvement_count = 0
max_iter_train = 10
max_iter_val = 10
tol_count = 10
optimizer = optim.Adam(model.parameters(), 
                       lr=lr, 
                       weight_decay=weight_decay)
if output_size_list[0] > 1:
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor([0.113,0.458,0.43])).to(device)
else:
    loss_fn = nn.BCEWithLogitsLoss()


In [ ]:
print("开始训练...")
for epoch in range(logger.start_epoch, EPOCHS):  # 从logger.start_epoch开始，而不是从0开始
    model.train()
    epoch_train_loss = 0.0
    
    # 创建训练数据加载器
    data_loaders = [dataset.iterate_batch(batch_size, normalize=True) for dataset in train_datasets]
    
    # 使用tqdm显示counter的进度
    pbar = tqdm(total=max_iter_train, desc=f"Epoch {epoch+1}/{EPOCHS}")
    counter = 1
    while counter <= max_iter_train:
        running_loss = 0.0
        y_true, y_pred = [], []
        
        # 获取每个数据加载器的下一批数据
        data_batches = []
        for loader_index, loader in enumerate(data_loaders):
            try:
                batch, ids = next(loader)
            except StopIteration:
                # 重新创建生成器
                loader = train_datasets[loader_index].iterate_batch(batch_size, normalize=True)
                batch, ids = next(loader)
                # 更新 data_loaders 列表中的生成器
                data_loaders[loader_index] = loader
            data_batches.append(batch)
        
        # 处理所有批次的数据
        for batch in data_batches:
            for i in range(len(batch)):  # 移除这里的tqdm
                datas = batch[i]
                
                lab, mask_lab, vit, mask_vit, ward_vit, mask_ward_vit, \
                                _, _, x_s, \
                                y_mat, y_mask, _, _ = datas
     
               
                # 将 lab 和 mask_lab 合并
                lab_c = torch.cat((lab, mask_lab), dim=-1).unsqueeze(0).to(device)
                
                # 将筛选后的 vit 和 mask_vit 合并，并添加维度
                vit_c = torch.cat((vit, mask_vit), dim=-1).unsqueeze(0).to(device)

                # 将筛选后的 vit 和 mask_vit 合并，并添加维度
                ward_vit_c = torch.cat((ward_vit, mask_ward_vit), dim=-1).unsqueeze(0).to(device)
                
                # 将 x_s 移动到设备
                x_s = x_s[:,:-4].to(device)

                for j in range(1, y_mat.shape[0]):
                    # 更新模型输入
                    inputs = [lab_c, ward_vit_c, vit_c[:,:j], x_s]
                
                    # 获取模型预测
                    yhat_list = model(inputs)
                    y_pred.append(yhat_list[0])
                    y_true.append(y_mat[j:(j+1),y_index:(y_index+1)])

        y_pred = torch.cat(y_pred, dim=0)
        y_true = torch.cat(y_true, dim=0)
        if output_size_list[0] == 1:
            y_true = (y_true > 0).float().to(device)
        else:
            y_true = y_true.to(torch.int64).to(device).reshape(-1)
        loss = loss_fn(y_pred, y_true)
        running_loss += loss.cpu().item()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_train_loss += running_loss
        
        # 更新进度条，并显示当前loss
        pbar.set_postfix({"Loss": f"{running_loss:.6f}"})
        pbar.update(1)
        
        counter += 1
    
    pbar.close()
    print(f"Epoch {epoch+1}/{EPOCHS} 完成, 平均训练损失: {epoch_train_loss/counter:.6f}")

    # 验证阶段
    print("开始在验证集上测试...")
    model.eval()
    running_loss = 0.0
    y_true, y_pred = [], []
    
    with torch.no_grad():
        # 创建验证数据加载器
        data_loaders = [dataset.iterate_batch(batch_size_val, normalize=True) for dataset in valid_datasets]
        
        # 使用tqdm显示验证进度
        pbar_val = tqdm(total=max_iter_val, desc="验证")
        counter = 1
        while counter <= max_iter_val:
            for loader in data_loaders:
                try:
                    data_batch, ids = next(loader)
                except StopIteration:
                    break

                for i in range(len(data_batch)):  # 移除这里的tqdm
                    datas = data_batch[i]
                    
                    lab, mask_lab, vit, mask_vit, ward_vit, mask_ward_vit, \
                                    _, _, x_s, \
                                    y_mat, y_mask, _, _ = datas
        
                
                    # 将 lab 和 mask_lab 合并
                    lab_c = torch.cat((lab, mask_lab), dim=-1).unsqueeze(0).to(device)
                    
                    # 将筛选后的 vit 和 mask_vit 合并，并添加维度
                    vit_c = torch.cat((vit, mask_vit), dim=-1).unsqueeze(0).to(device)

                    # 将筛选后的 vit 和 mask_vit 合并，并添加维度
                    ward_vit_c = torch.cat((ward_vit, mask_ward_vit), dim=-1).unsqueeze(0).to(device)
                    
                    # 将 x_s 移动到设备
                    x_s = x_s[:,:-4].to(device)
                    
                    for j in range(1, y_mat.shape[0]):
                        # 更新模型输入
                        inputs = [lab_c, ward_vit_c, vit_c[:,:j], x_s]
                    
                        # 获取模型预测
                        yhat_list = model(inputs)
                        y_pred.append(yhat_list[0])
                        y_true.append(y_mat[j:(j+1),y_index:(y_index+1)])

            pbar_val.update(1)
            counter += 1
        
        pbar_val.close()

        y_pred = torch.cat(y_pred, dim=0)
        y_true = torch.cat(y_true, dim=0)
        if output_size_list[0] == 1:
            y_true = (y_true > 0).float().to(device)
        else:
            y_true = y_true.to(torch.int64).to(device).reshape(-1)
        loss = loss_fn(y_pred, y_true)
        running_loss += loss.cpu().item()
        
        # 将模型输出转换为概率
        y_pred_prob = y_pred.cpu().numpy()
        y_true_np = y_true.cpu().numpy()

        # 计算AUC
        if output_size_list[0] == 1:
            auc = roc_auc_score(y_true_np, y_pred_prob)
        else:
            auc = roc_auc_score(y_true_np, y_pred_prob, average='macro', multi_class='ovr')
        print("Valid loss: {:.6f}, AUC: {:.6f}".format(running_loss, auc))

        # 使用记录器保存检查点
        is_best = logger.update_best_metrics(epoch, running_loss, auc)
        logger.save_checkpoint(
            model=model,
            epoch=epoch,
            train_loss=running_loss,
            valid_loss=running_loss,
            auc=auc,
            is_best=is_best,
            optimizer=optimizer
        )
        
        if is_best:
            no_improvement_count = 0
            print("模型已更新")
        else:
            no_improvement_count += 1
        
        if no_improvement_count == tol_count:
            print(f"Early stopping at epoch {epoch}")
            break


In [ ]:
# 在训练循环结束后（early stopping 或完成所有 epoch）添加测试代码
print("开始测试...")

# 创建测试数据集 - 修改为与训练集相同的格式
dataset_te = INSPIRE(
    "data/all_op_id/",
    ds[ds['dataset'] == 2],  # 测试集
    id_col="op_id",
    param_path="./param_folder"
)

# 使用logger加载最优模型
print("加载最优模型...")
model = logger.load_best_model(model)

running_loss = 0.0
y_true, y_pred = [], []
sample_ids = []  # 存储样本ID
time_points = []  # 存储对应的时间点

model.eval()  # 设置为评估模式
with torch.no_grad():
    # 使用dataset_te的样本数来展示进度
    for i in tqdm(range(dataset_te.len()), total=dataset_te.len(), desc="测试进度"):
        datas = dataset_te.get_1data(i, normalize=True)
        
        lab, mask_lab, vit, mask_vit, ward_vit, mask_ward_vit, \
                        _, t_list, x_s, \
                        y_mat, y_mask, y_static, y_mask1 = datas
        
        # 获取样本ID
        sample_id = dataset_te.all_id[i]
        
        # 将 lab 和 mask_lab 合并
        lab_c = torch.cat((lab, mask_lab), dim=-1).unsqueeze(0).to(device)
        
        # 将筛选后的 ward_vit 和 mask_ward_vit 合并，并添加维度
        ward_vit_c = torch.cat((ward_vit, mask_ward_vit), dim=-1).unsqueeze(0).to(device)
        
        # 将筛选后的 vit 和 mask_vit 合并，并添加维度
        vit_c = torch.cat((vit, mask_vit), dim=-1).unsqueeze(0).to(device)

        # 将 x_s 移动到设备
        x_s = x_s[:,:-4].to(device)
        
        for j in range(1, y_mat.shape[0]):
            # 更新模型输入
            inputs = [lab_c, ward_vit_c, vit_c[:,:j], x_s]
        
            # 获取模型预测
            yhat_list = model(inputs)
            y_pred.append(yhat_list[0])
            y_true.append(y_mat[j:(j+1),y_index:(y_index+1)])
            
            # 保存样本ID和对应的时间点
            sample_ids.append(sample_id)
            time_points.append(t_list[j].item())  # 保存当前时间点

    y_pred = torch.cat(y_pred, dim=0)
    y_true = torch.cat(y_true, dim=0)
    if output_size_list[0] == 1:
        y_true = (y_true > 0).float().to(device)
    else:
        y_true = y_true.to(torch.int64).reshape(-1).to(device)
    loss = loss_fn(y_pred, y_true)
    running_loss += loss.cpu().item()

# 将模型输出转换为概率
y_pred_prob = y_pred.cpu().numpy()  # 模型输出已经是概率了
y_true_np = y_true.cpu().numpy()

print(f"测试损失: {running_loss}")
print(logger.get_training_summary())  # 打印训练摘要信息


In [ ]:
# 创建DataFrame保存预测结果，包含时间点信息和所有类别的概率
results_dict = {
    'op_id': sample_ids,
    'time_point': time_points,  # 添加时间点信息
    'y_true': y_true_np.flatten()
}

# 添加每个类别的概率
for i in range(y_pred_prob.shape[1]):
    results_dict[f'y_pred_prob_{i}'] = y_pred_prob[:, i]

# 创建DataFrame
results_df = pd.DataFrame(results_dict)


In [ ]:
# 保存结果到CSV文件 - 修复datetime.now()的使用
from datetime import datetime  # 正确导入datetime类
results_path = f"results/{task_name}_test_intraop_minn.csv"
os.makedirs(os.path.dirname(results_path), exist_ok=True)
results_df.to_csv(results_path, index=False)
print(f"预测结果已保存到: {results_path}")
